# OPUS-MT en-tl — DIMER English→Tagalog translation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/tutorials/marianmt_translation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Helsinki--NLP%2Fopus--mt--en--tl-ffcc4d?style=flat)](https://huggingface.co/Helsinki-NLP/opus-mt-en-tl) [![Upstream](https://img.shields.io/badge/Upstream-Helsinki--NLP%2FOPUS--MT--train-181717?style=flat&logo=github&logoColor=white)](https://github.com/Helsinki-NLP/OPUS-MT-train) [![arXiv](https://img.shields.io/badge/arXiv-1804.00344-b31b1b.svg)](https://arxiv.org/abs/1804.00344)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** batched English→Tagalog machine translation (EN→TL only) using the pinned `Helsinki-NLP/opus-mt-en-tl` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/marianmt_translation_pipeline/pipeline.py` at revision `6cc90a3e344b`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `e46e1761492cb6a6fb9515a72bb55ca654815ca5` (~299 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

`Helsinki-NLP/opus-mt-en-tl` is a ~77 M-parameter OPUS-MT Marian transformer (6+6 layers, `d_model` 512) trained by the Helsinki-NLP group on the `opus+bt` corpus and released on 2020-02-26 — an **EN→TL only** model: English in, Tagalog out, and nothing else. At inference the encoder reads the English sentence once and the decoder emits one SentencePiece token per step until `</s>` or a step ceiling; **beam search with 4 beams** (the snapshot's `generation_config.json`) is the default decision rule and greedy decoding is available on request — there is no sampling and no seed. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. **The checkpoint is a pickle:** the only upstream weight file at this revision is `pytorch_model.bin` (no SafeTensors), so Section 3 re-hashes it against the inline SHA-256 manifest **before** it is opened, and the carried module then loads it with `use_safetensors=False, weights_only=True`, which makes `transformers` call `torch.load(weights_only=True)` — a restricted unpickler. What the upstream checkpoint supplies is the model and the SentencePiece tokenizer; what the carried pipeline module adds is manifest verification, input validation with named ceilings, batched translation with a fixed output contract, and the `validate_inputs` and `evaluation_report` stage helpers.

**Environment note:** `sacremoses` is not pinned and not installed, so `MarianTokenizer` prints one `Recommended: pip install sacremoses.` warning and uses the identity function as its source-side punctuation normaliser; the card-pass smoke ran in exactly that state and the outputs below are from it.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author three English sentences (or upload your own), stage and digest-verify the immutable upstream snapshot — a pickle checkpoint pinned by SHA-256 and loaded with `weights_only=True` — surface the pipeline's ceilings and the decision rule and validate the batch into an input manifest before any model work, translate through the public API with explicit `max_new_tokens`/`num_beams`, read `stopped_by` and the token counts correctly, read from the machine-readable evaluation report why **no metric is reported** and what references a BLEU/chrF evaluation would need, and export every translation with its identifier plus provenance.

**This notebook does not demonstrate:** Tagalog→English or any other direction, document-level translation with sentence splitting, instruction following or chat, sampling-based decoding, glossary or terminology control, source-side Moses punctuation normalisation (`sacremoses` is not installed), or any BLEU/chrF measurement. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float32 there too). CPU is adequate: the repository's model card records, for the Windows-venv smoke on an Intel Core Ultra 9 275HX, 6.0 s to load and digest-verify the 299 MB snapshot and 0.179 s (4 beams) / 0.108 s (greedy) for a batch of three sentences. The pinned `torch==2.14.0` install and the 296 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-decoder (seq2seq) model is; what beam search and greedy decoding do; why a translation can be fluent and wrong.
- **Data:** the default sample is **synthetic** — three English sentences authored in code (the ones the card-pass smoke used) — so nothing is downloaded and no private data is needed. It carries no reference translations, so any number it produces is smoke/sanity evidence, never a quality measurement. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction; each non-empty line of the uploaded UTF-8 text file is one English input. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Helsinki-NLP/opus-mt-en-tl` snapshot (~299 MB in total) at revision `e46e1761492c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'sentencepiece==0.2.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'marianmt-en-tl-translation-pipeline',
    'repository_revision': '6cc90a3e344b2b4f5f095cd29872682cec11d50a',
    'embedded_module': 'src/marianmt_translation_pipeline/pipeline.py',
    'embedded_modules': ['src/marianmt_translation_pipeline/pipeline.py'],
    'module_sha256': '0cf1d5bb099a87b6b68c017de57ddc597677f48eae2a5bd1d8a25f8d3c86ee29',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/marianmt_translation_pipeline/` @ `6cc90a3e344b`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/marianmt_translation_pipeline/pipeline.py`

In [ ]:
"""English-to-Tagalog machine translation with the pinned ``Helsinki-NLP/opus-mt-en-tl`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/opus-mt-en-tl/``) or, when
explicitly allowed, from the Hugging Face Hub at the pinned revision. The upstream snapshot ships its
weights as ``pytorch_model.bin`` — a pickle, not SafeTensors — so the trust boundary is the SHA-256 in the
manifest (checked before the load) plus weights_only=True deserialization in ``transformers``. One
task method, ``translate``: a batch of English strings in, one Tagalog string per input out.
Direction is EN -> TL only; the checkpoint has no reverse direction.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "Helsinki-NLP/opus-mt-en-tl"
MODEL_REVISION = "e46e1761492cb6a6fb9515a72bb55ca654815ca5"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "opus-mt-en-tl"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "pytorch_model.bin"  # the only upstream weight file at this revision: a pickle, digest-pinned

SOURCE_LANG = "en"  # tokenizer_config.json source_lang
TARGET_LANG = "tl"  # tokenizer_config.json target_lang
MAX_INPUT_TOKENS = 512  # max_position_embeddings in the snapshot config.json; longer inputs rejected, not cut
MAX_NEW_TOKENS = 512  # ceiling on decoder steps per call (config.json / generation_config.json max_length)
DEFAULT_MAX_NEW_TOKENS = 128
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard per input string
MAX_BATCH = 16  # texts per translate() call
MAX_NUM_BEAMS = 8
DEFAULT_NUM_BEAMS = 4  # num_beams in the snapshot generation_config.json
DECISION_RULE = (
    "beam search over whole sequences (num_beams=4 by default, from generation_config.json); greedy argmax "
    "per step when num_beams=1; no sampling; decoding stops at </s> or max_new_tokens"
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "sequence of non-empty English str (a sentence or short passage each); one Tagalog str each",
    "direction": f"{SOURCE_LANG}->{TARGET_LANG} only",
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "input_tokens": [1, MAX_INPUT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": [1, MAX_NUM_BEAMS],
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "Marian normalisation + SentencePiece encoding with source.spm (no truncation: an input over "
        "MAX_INPUT_TOKENS is rejected with a ValueError naming the count, never cut); batch padded to the "
        "longest input"
    ),
}


def _check_inputs(texts: Any, max_new_tokens: Any, num_beams: Any) -> list[str]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the texts as a list.

    The encoder-token ceiling is not checked here because it needs the loaded tokenizer;
    ``_check_input_tokens`` applies it inside the pipeline once the counts are known.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..MAX_BATCH={MAX_BATCH} items, got {len(texts)}")
    clean = []
    for i, text in enumerate(texts):
        if not isinstance(text, str):
            raise TypeError(f"texts[{i}] must be str, got {type(text).__name__}")
        if not text.strip():
            raise ValueError(f"texts[{i}] is empty")
        if len(text) > MAX_TEXT_CHARS:
            raise ValueError(f"texts[{i}] has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
        clean.append(text)
    for name, value, ceiling in (
        ("max_new_tokens", max_new_tokens, MAX_NEW_TOKENS),
        ("num_beams", num_beams, MAX_NUM_BEAMS),
    ):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an int")
        if not 1 <= value <= ceiling:
            raise ValueError(f"{name} must be between 1 and {ceiling}, got {value}")
    return clean


def _check_input_tokens(counts: Sequence[int]) -> list[int]:
    """The encoder-token ceiling, applied once the tokenizer has counted every input."""
    for i, n_input in enumerate(counts):
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(
                f"texts[{i}] is {n_input} tokens; ceiling is MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}"
            )
    return list(counts)


def validate_inputs(
    texts: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = DEFAULT_NUM_BEAMS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``translate`` would: both route through
    ``_check_inputs``. The encoder-token ceiling (``MAX_INPUT_TOKENS``) needs the loaded tokenizer and
    is enforced inside ``translate``, which reports ``input_tokens`` per item.
    """
    checked = _check_inputs(texts, max_new_tokens, num_beams)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"input{i:02d}", "chars": len(text), "words": len(text.split())}
            for i, text in enumerate(checked)
        ],
        "max_new_tokens": max_new_tokens,
        "num_beams": num_beams,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    The repository ships no metric helper, so the verdict is always ``not-measurable`` (EVAL9).
    ``references`` exists for interface parity with the fleet's other pipelines and is recorded in
    ``reason`` rather than scored: BLEU/chrF need a scorer and enough referenced sentences to state a
    dispersion, and manufacturing a number from a proxy such as length ratio would misrepresent a
    plumbing check as a quality measurement.
    """
    items = result.get("translations", [])
    rule = result.get("generation", {}).get("decision_rule", DECISION_RULE)
    supplied = references is not None
    return {
        "task": f"machine translation {SOURCE_LANG}->{TARGET_LANG}",
        "score_semantics": (
            "the pipeline emits no probability, confidence or score: generated_tokens, input_tokens and "
            f"stopped_by are counts and flags, and {rule} "
            "produces some token at every step with no minimum-probability cut-off and no shipped acceptance "
            "threshold"
        ),
        "sample_kind": sample_kind,
        "n_inputs": len(items),
        "n_generated_tokens": int(sum(int(item.get("generated_tokens", 0)) for item in items)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the repository ships no metric helper and a translation has no ground truth here"
            + (
                "; references were supplied but no metric helper exists to score them, and a handful of "
                "references is not a dispersion"
                if supplied
                else "; the evaluated sample has no reference translations"
            )
        ),
        "needs": (
            "reference Tagalog translations from the deployment domain, one or more per source sentence, "
            "over "
            "enough sentences to state a dispersion, scored with the caller's own BLEU/chrF implementation "
            "(the upstream README reports BLEU 26.6 / chrF 0.577 on Tatoeba.en.tl — an upstream claim, not "
            "measured here), excluding or re-running outputs whose stopped_by is max_new_tokens; no proxy "
            "such as length ratio substitutes for that"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class MarianMTTranslationPipeline:
    """``_runner(texts, max_new_tokens, num_beams)`` -> list of ``(translation, generated_tokens,
    stopped_by)``, one per input; ``_count_tokens(text)`` -> encoder token count incl. EOS. Both
    injectable so tests run offline."""

    _runner: Callable[[list[str], int, int], list[tuple[str, int, str]]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MarianMTTranslationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import MarianMTModel, MarianTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = MarianTokenizer.from_pretrained(location, trust_remote_code=False, **kwargs)
        # Trust boundary (MOD12): the upstream weight file is a pickle (pytorch_model.bin). Its SHA-256 was
        # checked against the manifest above; use_safetensors=False names that fact, and weights_only=True
        # makes transformers deserialise with weights_only=True, which refuses arbitrary objects.
        model = MarianMTModel.from_pretrained(
            location,
            dtype=torch.float32,
            trust_remote_code=False,
            use_safetensors=False,
            weights_only=True,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        eos_id, pad_id = model.config.eos_token_id, model.config.pad_token_id

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, truncation=False)["input_ids"])

        def runner(texts: list[str], max_new_tokens: int, num_beams: int) -> list[tuple[str, int, str]]:
            enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=False).to(resolved_device)
            with torch.inference_mode():
                out = model.generate(
                    **enc, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            results = []
            for row in out.tolist():
                content = [t for t in row if t not in (eos_id, pad_id)]
                stopped_by = "eos" if eos_id in row else "max_new_tokens"
                results.append(
                    (tokenizer.decode(content, skip_special_tokens=True), len(content), stopped_by)
                )
            return results

        return cls(runner, count_tokens, resolved_device, source)

    def _validate(self, texts: Any, max_new_tokens: Any, num_beams: Any) -> tuple[list[str], list[int]]:
        clean = _check_inputs(texts, max_new_tokens, num_beams)
        return clean, _check_input_tokens([self._count_tokens(text) for text in clean])

    def translate(
        self,
        texts: Sequence[str],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = DEFAULT_NUM_BEAMS,
    ) -> dict[str, Any]:
        """Translate a batch of English texts to Tagalog; one output per input, in order."""
        clean, counts = self._validate(texts, max_new_tokens, num_beams)
        generated = self._runner(clean, max_new_tokens, num_beams)
        if len(generated) != len(clean):
            raise RuntimeError(f"runner returned {len(generated)} outputs for {len(clean)} inputs")
        translations = []
        for text, n_input, (output, n_generated, stopped_by) in zip(clean, counts, generated, strict=True):
            if not isinstance(output, str) or not isinstance(n_generated, int):
                raise RuntimeError("runner must return (str, int, str) per input")
            translations.append(
                {
                    "source": text,
                    "text": output,
                    "input_tokens": n_input,
                    "generated_tokens": n_generated,
                    "stopped_by": stopped_by,
                }
            )
        return {
            "translations": translations,
            "n": len(translations),
            "direction": f"{SOURCE_LANG}->{TARGET_LANG}",
            "generation": {
                "max_new_tokens": max_new_tokens,
                "num_beams": num_beams,
                "do_sample": False,
                "decision_rule": DECISION_RULE,
            },
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e46e1761492c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MarianMTTranslationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "opus-mt-en-tl",
  "modelId": "Helsinki-NLP/opus-mt-en-tl",
  "revision": "e46e1761492cb6a6fb9515a72bb55ca654815ca5",
  "files": [
    {
      "path": "README.md",
      "bytes": 839,
      "sha256": "debb5b41e6d8c18590466ce0c3b9bcfc3634f14454cac8f66b71dd3dcb36a58e"
    },
    {
      "path": "config.json",
      "bytes": 1361,
      "sha256": "f7c975ff86205d2e19e4a2fa93dd8cd124a707ab567f4044173fd9cda895d1aa"
    },
    {
      "path": "generation_config.json",
      "bytes": 293,
      "sha256": "2e6d06b221eee26979f89f53735306f4e3d927808cc61a062218936729c4329a"
    },
    {
      "path": "pytorch_model.bin",
      "bytes": 296434867,
      "sha256": "e418d573a717b2c81eaa3a80c8eb68f203c14c6b734dd36d41708c1369d0eb44"
    },
    {
      "path": "source.spm",
      "bytes": 826681,
      "sha256": "feaf3e0cf579e3daeb6bac5dc94b906efed40e89b544d269cec9bdbb4f05f8e1"
    },
    {
      "path": "target.spm",
      "bytes": 834725,
      "sha256": "e26d20a4aae1e82aaab382ff47ad8b587d03d1087e295d4c51ff71d63427e059"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 42,
      "sha256": "5ee0a3aea386b802c612ef5cce6f77fa4271ccb01b64dce0e4aca14725203ee1"
    },
    {
      "path": "vocab.json",
      "bytes": 1324714,
      "sha256": "43bf29e5d430061afa7d978d68c09683dc95650959607ae74eaaf5471285df57"
    }
  ],
  "totalBytes": 299423522
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MarianMTTranslationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: three English sentences authored in this cell — the same three the repository's card-pass smoke translated (`The house is wonderful.`, `Good morning to all of you.`, `Where is the nearest hospital?`). None has a reference translation, so nothing in this notebook is a quality measurement; the card's smoke observations (`Ang bahay ay kahanga - hanga.`, `Magandang umaga sa inyong lahat.`, `Nasaan ang pinakamalapit na ospital?`) are one run on one machine, not expected values this notebook asserts. The sample identity and a SHA-256 of its text are printed so an export can be tied to exactly these inputs.

Two Colab form parameters fix the generation settings for the whole batch: `GEN_MAX_NEW_TOKENS` (default 128, the package's `DEFAULT_MAX_NEW_TOKENS`) and `NUM_BEAMS` (default 4, the snapshot's `generation_config.json` value and the package's `DEFAULT_NUM_BEAMS`; set 1 for greedy). They are checked against the carried module's ceilings in the next section.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file in which every non-empty line is one **English** input; at most `MAX_BATCH` lines per run, each at most `MAX_TEXT_CHARS` characters and tokenising to at most `MAX_INPUT_TOKENS` SentencePiece pieces, which the pipeline enforces by rejecting, not by truncating. The upload stays inside this runtime. If you hold reference Tagalog translations for your lines, keep them outside the notebook — Section 7 explains what to compute with them.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
GEN_MAX_NEW_TOKENS = 128  # @param {type:"integer"}
NUM_BEAMS = 4  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    texts = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if not texts:
        raise ValueError(f'{sample_name}: expected at least one non-empty English line')
    sample_kind = 'BYOD upload'
else:
    texts = [
        'The house is wonderful.',
        'Good morning to all of you.',
        'Where is the nearest hospital?',
    ]
    sample_name = 'synthetic_english_sentences'
    sample_kind = 'synthetic (authored in this cell)'
item_ids = [f'input{index:02d}' for index in range(len(texts))]
sample_sha256 = hashlib.sha256('\n'.join(texts).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'inputs': len(texts), 'text_sha256': sample_sha256, 'max_new_tokens': GEN_MAX_NEW_TOKENS, 'num_beams': NUM_BEAMS})
for item_id, text in zip(item_ids, texts, strict=True):
    print(f'{item_id}: {text[:110]}' + ('...' if len(text) > 110 else ''))

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `translate` applies — both route through the same private `_check_inputs` — so the sequence type, the batch size 1..`MAX_BATCH`, each item's type, non-emptiness and character ceiling `MAX_TEXT_CHARS`, `max_new_tokens` in 1..`MAX_NEW_TOKENS` and `num_beams` in 1..`MAX_NUM_BEAMS` are enforced identically. It returns one **input manifest** naming the schema and ceilings, the direction (`en->tl only`), each input's identifier, character and word counts, the settings in force, and the verdict; it is written to `outputs/marianmt_translation_input_manifest.json`. `MAX_INPUT_TOKENS` (encoder tokens including `</s>`; the upstream `max_position_embeddings`) needs the real tokenizer and is therefore enforced inside `translate`, which **rejects with a `ValueError` naming the count, never silently cuts**; every translation reports `input_tokens`. `DECISION_RULE` states the decoding rule in force (beam search, 4 beams by default; greedy when `num_beams=1`; no sampling). To show what rejection looks like, the cell also validates an out-of-range `num_beams` and records the pipeline's own error message as a finding. Nothing here trims or alters the texts, and nothing checks that they are English — a non-English line is accepted and mistranslated.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_BATCH': MAX_BATCH, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_NUM_BEAMS': DEFAULT_NUM_BEAMS}
print(ceilings)
print({'decision_rule': DECISION_RULE, 'direction': f'{SOURCE_LANG}->{TARGET_LANG}'})
input_manifest = validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, names=item_ids)
# Demonstrate rejection on a setting that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=MAX_NUM_BEAMS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'num-beams-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/marianmt_translation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceiling': 'enforced by translate() with the real tokenizer; reported as input_tokens per item'})

## 6. Translate and read the outputs correctly

`translate(texts, max_new_tokens=..., num_beams=...)` runs the whole batch through the encoder-decoder in one padded pass and returns a dict with one `translations` entry per input, in order: `source` (the English input), `text` (the decoded Tagalog with special tokens removed), `input_tokens` (encoder tokens including `</s>`, the number checked against `MAX_INPUT_TOKENS`), `generated_tokens` (decoder tokens emitted, excluding `</s>`/pad) and `stopped_by` (`eos` when the model ended the sequence itself, `max_new_tokens` when it hit the ceiling — such an output is cut mid-sentence and should be re-run with a larger `GEN_MAX_NEW_TOKENS` before anyone reads it as a finished translation); plus `n`, `direction`, the `generation` settings actually used (`max_new_tokens`, `num_beams`, `do_sample=False`, `decision_rule`), `device`, `source` and the model identity. **Score semantics:** the pipeline emits **no probability, confidence or score of any kind** — the counts above are counts, not scores; beam search keeps the highest-scoring sequences with no minimum-probability cut-off, so some token is always produced, and the pipeline ships no acceptance threshold on output quality. Whoever deploys it owns any acceptance rule, judged on their own references. The run is deterministic for a given batch, settings, weights, device and library versions (no sampling, `model.eval()`, no seed needed); batch padding and float32 kernel differences between CPU and CUDA can flip a near-tied token and change the rest of the sequence from that point, and beam search can differ from greedy. The checks below are falsifiable plumbing checks — one result per input, in order, every count within its ceiling, the direction echoed — plus the batch wall time measured on the runtime identified in Section 1 (includes warm-up). Look for three short Tagalog sentences; whether they are *good* is exactly what no number here can tell you.

In [ ]:
import time

started = time.perf_counter()
result = pipe.translate(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
elapsed = time.perf_counter() - started
results = [{'id': item_id, 'seconds_batch': round(elapsed, 3), **item} for item_id, item in zip(item_ids, result['translations'], strict=True)]
for r in results:
    print(f"{r['id']} {r['input_tokens']} -> {r['generated_tokens']} tokens, stopped_by={r['stopped_by']}")
    print(f"    {r['source']}")
    print(f"    {r['text']}")
print({'batch_seconds': round(elapsed, 3), 'direction': result['direction'], 'device': result['device']})
checks = {
    'one_result_per_input_in_order': [r['source'] for r in results] == list(texts),
    'generated_within_ceiling': all(r['generated_tokens'] <= GEN_MAX_NEW_TOKENS for r in results),
    'input_within_ceiling': all(r['input_tokens'] <= MAX_INPUT_TOKENS for r in results),
    'direction_is_en_to_tl': result['direction'] == f'{SOURCE_LANG}->{TARGET_LANG}',
    'settings_echoed': result['generation']['max_new_tokens'] == GEN_MAX_NEW_TOKENS and result['generation']['num_beams'] == NUM_BEAMS and result['generation']['do_sample'] is False,
}
if not all(checks.values()):
    raise RuntimeError(f'translate output failed a sanity check: {checks}')
print({'checks': checks, 'decision_rule': result['generation']['decision_rule'], 'hit_token_ceiling': [r['id'] for r in results if r['stopped_by'] == 'max_new_tokens']})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. The repository ships **no metric helper and reports no performance measure**: machine translation is conventionally scored with BLEU and chrF (or COMET) against human **reference translations** — one or more Tagalog references per source sentence — over enough sentences to state a dispersion, and the synthetic sample has none, so the verdict is always `not-measurable` and none is manufactured from a proxy such as length ratio. Supplying references does not change the verdict, because no metric helper exists to score them and a handful of references is not a dispersion; the helper records that in `reason`. The report covers the whole batch (`n_inputs`, summed `n_generated_tokens`) and lands at `outputs/marianmt_translation_evaluation_report.json`. The upstream Tatoeba figures in the model card (BLEU 26.6, chr-F 0.577) are upstream claims, not measured here.

In [ ]:
report = evaluation_report(result, sample_kind=sample_kind)
with open('outputs/marianmt_translation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: the sample has no reference translations, and the repository ships no metric helper; compute BLEU/chrF on your own referenced sentences.')

## 8. Export the translations and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report: `outputs/marianmt_translation_translations.csv` — one row per input with its identifier, the English source, the Tagalog output, both token counts, `stopped_by` and the batch wall time, so every translation maps back to its input — and `outputs/marianmt_translation_result.json`, which carries the same items plus the generation settings in force, the direction, the ceilings, the sanity checks, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary (including the pickle's manifest digest, so the export records what was loaded), and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

items = [
    {'id': r['id'], 'source': r['source'], 'translation': r['text'], 'input_tokens': r['input_tokens'], 'generated_tokens': r['generated_tokens'], 'stopped_by': r['stopped_by'], 'seconds_batch': r['seconds_batch']}
    for r in results
]
with open('outputs/marianmt_translation_translations.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(items[0]))
    writer.writeheader()
    writer.writerows(items)
weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
payload = {
    'items': items,
    'generation': result['generation'],
    'direction': result['direction'],
    'ceilings': ceilings,
    'sanity_checks': checks,
    'translations_file': 'outputs/marianmt_translation_translations.csv',
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'inputs': len(texts), 'text_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'pytorch pickle, digest-verified, loaded with weights_only=True', 'weight_sha256': weight_entry['sha256']},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
        'source': pipe.source,
    },
}
with open('outputs/marianmt_translation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The Tagalog strings are the model's beam-search (or greedy) output for *your* English input: fluent text that can drop a negation, change a number, leave an entity in English or pick a politeness register the source never specified, and the pipeline attaches no probability, confidence or quality score to it — `generated_tokens`, `input_tokens` and `stopped_by` are counts and flags, not evidence of adequacy. On the synthetic sample the checks prove only that the input contract, the digest-verified pickle load, the batched generation path and the ordering work end to end; the evaluation report is `not-measurable` because nothing can be computed without reference translations, and a real evaluation needs referenced sentences from your own domain, a BLEU/chrF-style scorer, and enough items to state a dispersion. Inputs above `MAX_INPUT_TOKENS` are refused rather than cut; multi-sentence inputs are translated as one sequence; outputs that stop at `max_new_tokens` are truncated mid-sentence; non-English input is accepted and mistranslated. The pipeline exposes no reverse direction, sentence splitting, sampling or terminology control, and `sacremoses` punctuation normalisation is not applied. Decoding is deterministic on a fixed device, dtype and batch, but CPU and CUDA float32 kernels can diverge on a near-tied token.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot (including its pickle weight file), validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, translation quality on any domain, a usable acceptance threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file (most likely the 296 MB `pytorch_model.bin`) is incomplete or altered — delete it from `weights/opus-mt-en-tl/` and rerun Section 3; the load never proceeds on a digest mismatch. `UserWarning: Recommended: pip install sacremoses.` in Section 3: expected; the normaliser is the identity in this configuration. `ValueError: texts[i] is N tokens; ceiling is MAX_INPUT_TOKENS=512` in Section 6: split or shorten that BYOD line and rerun from Section 4. `stopped_by` equal to `max_new_tokens`: raise `GEN_MAX_NEW_TOKENS` (ceiling `MAX_NEW_TOKENS`) and rerun Section 6. Output that repeats or echoes the input: the line is probably not English.

**Next experiments.** Set `NUM_BEAMS = 1` and compare the greedy outputs with the 4-beam ones (the card-pass smoke found the three sentences unchanged); translate the same sentence with curly quotes and with straight quotes and compare, since no punctuation normalisation runs; hand the pipeline a sentence you hold a human Tagalog reference for and score the output with a BLEU/chrF implementation of your choice — the first step towards the real evaluation the report asks for; run the same batch on a CUDA runtime and diff the outputs against the CPU run. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and the pickle trust boundary: https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Helsinki-NLP/opus-mt-en-tl
- Upstream training code: https://github.com/Helsinki-NLP/OPUS-MT-train
- OPUS-MT — Building open translation services for the World (Tiedemann & Thottingal, EAMT 2020): https://aclanthology.org/2020.eamt-1.61
- Marian: Fast Neural Machine Translation in C++ (Junczys-Dowmunt et al., ACL 2018): https://arxiv.org/abs/1804.00344